# COMPACT-MI-RFE — Colab Pro
Stage 6 setup + Stage 7.1 SVM technical probe. Chạy Cell 1–5 theo thứ tự (gồm Cell 4A tự đồng bộ mã hỗ trợ), sau đó kiểm tra hoặc khôi phục dataset ở Cell 6. Mặc định chỉ dry run; hai kết quả RF FS16 không chạy lại. Không dùng Test trước khi khóa lựa chọn và refit; full SVM vẫn bị chặn.


In [ ]:
# CELL 1 — Runtime information (read-only; no GPU acceleration assumed)
import os, platform, shutil, sys, importlib.metadata as im
from pathlib import Path
try:
    import psutil
    vm = psutil.virtual_memory()
    ram = {"total": vm.total, "available": vm.available}
    physical = psutil.cpu_count(logical=False)
except ImportError:
    ram, physical = {"total": None, "available": None}, None
disk = shutil.disk_usage("/content")
versions = {}
for name in ("numpy","pandas","scipy","scikit-learn","joblib","psutil"):
    try: versions[name] = im.version(name)
    except im.PackageNotFoundError: versions[name] = None
gpu_tool = shutil.which("nvidia-smi")
print({"os":platform.platform(),"arch":platform.machine(),"python":sys.version,
       "logical_cpus":os.cpu_count(),"physical_cpus":physical,"ram_bytes":ram,
       "disk_free_bytes":disk.free,"gpu_probe_available":bool(gpu_tool),"versions":versions})
print("This scikit-learn pipeline runs on CPU; Colab Pro/High-RAM is not assumed.")

{'os': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'arch': 'x86_64', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'logical_cpus': 8, 'physical_cpus': 4, 'ram_bytes': {'total': 54750404608, 'available': 53306941440}, 'disk_free_bytes': 207569371136, 'gpu_probe_available': True, 'versions': {'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'joblib': '1.6.0', 'psutil': '5.9.5'}}
This scikit-learn pipeline runs on CPU; Colab Pro/High-RAM is not assumed.


In [ ]:
# CELL 2 — Mount Google Drive (you authorize the Google prompt)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted:", Path('/content/drive/MyDrive').is_dir())

Mounted at /content/drive
Drive mounted: True


In [ ]:

# CELL 3 — Load BOTH datasets from mounted Google Drive
# WINDOWS_CPLUS + CSV_SPLIT_CPLUS
# No download, no copy, no extraction

from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

# 2. Configure both dataset paths
DRIVE_DATASET_PATH = "/content/drive/MyDrive/WINDOWS_CPLUS"

CSV_SPLIT_DATASET_PATH = "/content/drive/MyDrive/CSV_SPLIT_CPLUS"

# Compatibility with existing Stage 6 notebook
SHARED_FOLDER_ID = ""
DRIVE_ARCHIVE_ID = ""

# 3. Resolve dataset folders
WINDOWS_SOURCE = Path(DRIVE_DATASET_PATH)
CSV_SPLIT_SOURCE = Path(CSV_SPLIT_DATASET_PATH)

# Handle nested directories if present
if (WINDOWS_SOURCE / "WINDOWS_CPLUS").is_dir():
    WINDOWS_SOURCE = WINDOWS_SOURCE / "WINDOWS_CPLUS"

if (CSV_SPLIT_SOURCE / "CSV_SPLIT_CPLUS").is_dir():
    CSV_SPLIT_SOURCE = CSV_SPLIT_SOURCE / "CSV_SPLIT_CPLUS"

# Update paths for downstream cells
DRIVE_DATASET_PATH = str(WINDOWS_SOURCE)
CSV_SPLIT_DATASET_PATH = str(CSV_SPLIT_SOURCE)

# 4. Verify both datasets
print("=== STAGE 6 — DATASET SOURCES ===")

for name, path in [
    ("WINDOWS_CPLUS", WINDOWS_SOURCE),
    ("CSV_SPLIT_CPLUS", CSV_SPLIT_SOURCE),
]:
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.is_dir())

    if not path.is_dir():
        raise FileNotFoundError(
            f"Dataset not found: {path}"
        )

    print("Contents:")
    for item in list(path.iterdir())[:10]:
        print("  ", item.name)

print("\n================================")
print("BOTH DATASETS FOUND: PASS")
print("No dataset copied or downloaded.")
print("================================")


=== STAGE 6 — DATASET SOURCES ===

WINDOWS_CPLUS
Path: /content/drive/MyDrive/WINDOWS_CPLUS
Exists: True
Contents:
   ctu13_sha256_before.json
   input_sha256_before.json
   resume_record.json
   resume_record_attempt_02.json
   reference_stat_samples.csv
   input_sha256_after.json
   ctu13_sha256_after.json
   window_run_summary.json
   progress.json
   FW_512_FS_16

CSV_SPLIT_CPLUS
Path: /content/drive/MyDrive/CSV_SPLIT_CPLUS
Exists: True
Contents:
   Cplus_cutoffs_used.csv
   source_manifest.csv
   split_manifest.csv
   proxy_label_counts_all_roles.csv
   label_mapping.csv
   scenario_counts_all_roles.csv
   validation.json
   test
   val
   train

BOTH DATASETS FOUND: PASS
No dataset copied or downloaded.


In [ ]:
# CELL 4 — Clone safely; existing checkout is inspected, never reset/pulled
import subprocess

REPO = Path('/content/RFforNetwork')
URL = 'https://github.com/kasum00/RFforNetwork.git'
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise FileExistsError(f'Not a Git checkout: {REPO}')
    status = subprocess.run(
        ['git', '-C', str(REPO), 'status', '--porcelain'],
        text=True, capture_output=True, check=True,
    )
    print('Existing checkout; no pull. Local changes:', status.stdout.splitlines())
else:
    subprocess.run(['git', 'clone', '--', URL, str(REPO)], check=True)

required = [
    'huyen_code/ciciomt24_training.py',
    'huyen_code/ciciomt24_sliding_window.py',
    'huyen_code/ciciomt24_split.py',
    'huyen_code/test_ciciomt24_training.py',
    'huyen_code/test_ciciomt24_split.py',
    'huyen_code/test_ciciomt24_sliding_window.py',
    # Cell 4A supplies the Stage 7.1 experiment_launcher.py.
    'configs/cplus_manifest/split_manifest.csv',
    'CODE_ctu13/[[CTU13]9_ex2_rf.py',
]
missing = [path for path in required if not (REPO / path).is_file()]
if missing:
    raise RuntimeError(
        f'GitHub checkout lacks Stage 6/Stage 5 files: {missing}. '
        'Publish selected source files manually; do not use an incomplete clone.'
    )
print('Repository ready:', REPO)

Existing checkout; no pull. Local changes: []
Repository ready: /content/RFforNetwork


## Stage 7.1 — Đồng bộ mã hỗ trợ tự động từ notebook

Notebook này chứa bản kiểm thử Stage 6 sửa lỗi 12/15 cell và ba file Stage 7.1 cho SVM. Sau Cell 4, chạy Cell 4A: cell tự đồng bộ bốn file hỗ trợ và đưa Cell 8 của notebook trong checkout về mặc định `dry_run`/confirmation rỗng; mọi bản bị thay đổi đều được sao lưu theo SHA256. Bản notebook bạn đang mở và kết quả đã chạy không bị ghi đè. Sau đó chạy Cell 5. Không upload CSV, không push Git, không chạy full SVM ở Stage 7.1. Nếu 323 mục dataset đã nằm trong `.incomplete`, dùng RECOVER CELL 6 thay vì chuyển lại dữ liệu.


In [ ]:
# CELL 4A — Embedded Stage 6 test fix + Stage 7.1 SVM support
# Writes only four support files in the ephemeral Colab checkout.
# No dataset transfer, model fit, or original CTU-13 source change.
from pathlib import Path
import base64
import gzip
import hashlib
import json
import os
import tempfile

repo = Path(REPO).resolve()
if repo != Path('/content/RFforNetwork') or not (repo / '.git').is_dir():
    raise RuntimeError('Run Cell 4 first; unexpected checkout path')

package_b64 = (
    '...'
)
package_sha256 = 'e744f9cbf02c9add6db03bec296b2ae4ece8d3cd6104452d8125f5909cae6a28'
package = base64.b64decode(package_b64, validate=True)
if hashlib.sha256(package).hexdigest() != package_sha256:
    raise ValueError('Embedded support package SHA256 mismatch')

records = json.loads(gzip.decompress(package))
expected_paths = {
    'huyen_code/ciciomt24_svm_probe.py',
    'huyen_code/test_ciciomt24_svm_probe.py',
    'colab/experiment_launcher.py',
    'colab/test_stage6_colab.py',
}
if set(records) != expected_paths:
    raise ValueError('Embedded support package file list mismatch')

decoded = {}
for relative_path in sorted(expected_paths):
    record = records[relative_path]
    data = base64.b64decode(record['base64'], validate=True)
    if hashlib.sha256(data).hexdigest() != record['sha256']:
        raise ValueError('Embedded file SHA256 mismatch: ' + relative_path)
    compile(data.decode('utf-8'), relative_path, 'exec')
    decoded[relative_path] = data

backup_root = Path('/content/stage71_source_backup')
for relative_path in sorted(expected_paths):
    destination = repo / relative_path
    if not destination.parent.is_dir() or destination.is_symlink():
        raise RuntimeError('Unsafe checkout target: ' + str(destination))
    wanted = decoded[relative_path]
    if destination.is_file() and destination.read_bytes() == wanted:
        print('Already current:', relative_path)
        continue
    if destination.exists() and not destination.is_file():
        raise RuntimeError('Not a regular checkout file: ' + str(destination))
    if destination.is_file():
        original = destination.read_bytes()
        old_sha = hashlib.sha256(original).hexdigest()
        backup = backup_root / (relative_path + '.' + old_sha[:16] + '.bak')
        backup.parent.mkdir(parents=True, exist_ok=True)
        if backup.exists():
            if backup.read_bytes() != original:
                raise FileExistsError('Backup collision: ' + str(backup))
        else:
            with backup.open('xb') as stream:
                stream.write(original)
        print('Backup:', backup)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode='wb', dir=str(destination.parent),
            prefix=destination.name + '.', suffix='.new', delete=False
        ) as stream:
            temporary = Path(stream.name)
            stream.write(wanted)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, destination)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    if hashlib.sha256(destination.read_bytes()).hexdigest() != records[relative_path]['sha256']:
        raise IOError('Checkout write verification failed: ' + relative_path)
    print('Installed:', relative_path)

# A reused Colab runtime may still cache the old launcher module.
import importlib
import sys
for module_name in ('colab.experiment_launcher', 'ciciomt24_svm_probe', 'test_ciciomt24_svm_probe'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

# The Stage 6 test reads the checkout notebook, not this open Colab tab.
# A prior RF run may have saved MODE=full_train there; restore its safe default.
import ast
import re
checkout_notebook = repo / 'notebooks/COMPACT_MI_RFE_ColabPro.ipynb'
if checkout_notebook.is_symlink() or not checkout_notebook.is_file():
    raise FileNotFoundError('Checkout notebook missing or unsafe: ' + str(checkout_notebook))
checkout_original = checkout_notebook.read_bytes()
checkout_document = json.loads(checkout_original)
checkout_cells = [cell for cell in checkout_document['cells']
                  if (cell.get('id') or cell.get('metadata', {}).get('id')) == 'stage6-cell-8']
if len(checkout_cells) != 1 or not isinstance(checkout_cells[0]['source'], list):
    raise ValueError('Checkout must contain exactly one Stage 6 Cell 8 with source lines')
checkout_lines = checkout_cells[0]['source']
mode_indices = [i for i, line in enumerate(checkout_lines) if re.match(r'^\s*MODE\s*=', line)]
confirmation_indices = [i for i, line in enumerate(checkout_lines)
                        if re.match(r'^\s*CONFIRMATION\s*=', line)]
if len(mode_indices) != 1 or len(confirmation_indices) != 1:
    raise ValueError('Checkout Cell 8 MODE/CONFIRMATION assignment is ambiguous')
mode_index, confirmation_index = mode_indices[0], confirmation_indices[0]
parsed = ast.parse(''.join(checkout_lines))
assignments = {target.id: node.value for node in parsed.body
               if isinstance(node, ast.Assign) for target in node.targets
               if isinstance(target, ast.Name) and target.id in ('MODE', 'CONFIRMATION')}
if set(assignments) != {'MODE', 'CONFIRMATION'}:
    raise ValueError('Checkout Cell 8 assignments could not be verified')
if not isinstance(assignments['MODE'], ast.Constant) or assignments['MODE'].value not in ('dry_run', 'full_train'):
    raise ValueError('Unexpected checkout MODE value')
if not isinstance(assignments['CONFIRMATION'], ast.Constant) or not isinstance(assignments['CONFIRMATION'].value, str):
    raise ValueError('Unexpected checkout CONFIRMATION value')
source_text = ''.join(checkout_lines)
mode_literal = ast.get_source_segment(source_text, assignments['MODE'])
confirmation_literal = ast.get_source_segment(source_text, assignments['CONFIRMATION'])
if not mode_literal or not confirmation_literal:
    raise ValueError('Checkout Cell 8 literal could not be located')
new_mode = checkout_lines[mode_index].replace(mode_literal, '"dry_run"', 1)
new_confirmation = checkout_lines[confirmation_index].replace(confirmation_literal, '""', 1)
if assignments['MODE'].value != 'dry_run' and new_mode == checkout_lines[mode_index]:
    raise ValueError('Checkout MODE replacement failed')
if assignments['CONFIRMATION'].value and new_confirmation == checkout_lines[confirmation_index]:
    raise ValueError('Checkout CONFIRMATION replacement failed')
if new_mode != checkout_lines[mode_index] or new_confirmation != checkout_lines[confirmation_index]:
    checkout_lines[mode_index] = new_mode
    checkout_lines[confirmation_index] = new_confirmation
    ast.parse(''.join(checkout_lines))
    original_sha = hashlib.sha256(checkout_original).hexdigest()
    notebook_backup = backup_root / ('notebooks/COMPACT_MI_RFE_ColabPro.ipynb.'
                                     + original_sha[:16] + '.bak')
    notebook_backup.parent.mkdir(parents=True, exist_ok=True)
    if notebook_backup.exists():
        if notebook_backup.read_bytes() != checkout_original:
            raise FileExistsError('Checkout notebook backup collision')
    else:
        with notebook_backup.open('xb') as stream:
            stream.write(checkout_original)
    updated = (json.dumps(checkout_document, ensure_ascii=False, indent=2) + '\n').encode('utf-8')
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='wb', dir=str(checkout_notebook.parent),
                                         prefix=checkout_notebook.name + '.',
                                         suffix='.new', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(updated)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, checkout_notebook)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    print('Checkout notebook Cell 8 restored to dry_run; backup:', notebook_backup)
else:
    print('Checkout notebook Cell 8 already has a safe dry_run default')

STAGE71_SOURCE_READY = all(
    hashlib.sha256((repo / rel).read_bytes()).hexdigest() == records[rel]['sha256']
    for rel in expected_paths
)
if not STAGE71_SOURCE_READY:
    raise RuntimeError('Support source verification failed')
print('CELL 4A: PASS — support files and checkout dry-run default verified; now run Cell 5.')

In [ ]:

# CELL 5 — Install dependencies, verify source and run tests
# Compatible with Python 3.12 / 3.13
# No dataset download. No training started.

from pathlib import Path
import importlib.metadata as im
import subprocess
import hashlib
import sys

# ============================================================
# 1. REPOSITORY
# ============================================================

REPO = Path("/content/RFforNetwork").resolve()

if not REPO.is_dir():
    raise FileNotFoundError(REPO)

if not globals().get('STAGE71_SOURCE_READY', False):
    raise RuntimeError('Chạy Cell 4A trong notebook này trước Cell 5 để đồng bộ bản test và mã SVM; không cần upload file Python.')

sys.path.insert(0, str(REPO))

from colab.setup_colab import (
    selected_versions,
    dependency_plan,
    repo_inventory,
)

print("Actual Python:", sys.version)
print("Repository:", REPO)

# ============================================================
# 2. SELECT SCIENTIFIC PROFILE
# ============================================================

expected = selected_versions()
print("\nSelected scientific profile:", expected)

def installed_version(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return None

current = {
    name: installed_version(name)
    for name in expected
}

extras = (
    "psutil",
    "google-api-python-client",
    "google-auth",
)

extras_missing = [
    name for name in extras
    if installed_version(name) is None
]

science_needs = any(
    current[name] != version
    for name, version in expected.items()
)

needs = science_needs or bool(extras_missing)

# ============================================================
# 3. INSTALL ONLY IF NEEDED
# ============================================================

requirements = REPO / "requirements-colab.txt"

if not requirements.is_file():
    raise FileNotFoundError(requirements)

if needs:
    print("\nInstalling missing/mismatched packages...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(requirements),
        ],
        check=True,
    )

    if science_needs:
        raise RuntimeError(
            "Scientific packages installed. "
            "Restart Colab session, then rerun Cells 1-5."
        )

# ============================================================
# 4. VERIFY EXACT SCIENTIFIC VERSIONS
# ============================================================

print("\n=== SCIENTIFIC VERSION CHECK ===")

for name, required in expected.items():
    actual = installed_version(name)

    status = (
        "PASS"
        if actual == required
        else "FAIL"
    )

    print(
        f"{name}: {actual} "
        f"| required={required} | {status}"
    )

    if actual != required:
        raise RuntimeError(
            f"Version mismatch: {name}"
        )

for name in extras:
    if installed_version(name) is None:
        raise RuntimeError(
            f"Missing package: {name}"
        )

# ============================================================
# 5. GLOBAL PIP DEPENDENCY AUDIT
# ============================================================

print("\n=== GLOBAL DEPENDENCY AUDIT ===")

pip_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "check",
    ],
    capture_output=True,
    text=True,
    check=False,
)

GLOBAL_PIP_CHECK_PASSED = (
    pip_result.returncode == 0
)

if GLOBAL_PIP_CHECK_PASSED:
    print("Global pip check: PASS")
else:
    print("WARNING: Global dependency conflicts detected.")
    print(pip_result.stdout)
    print(pip_result.stderr)

    # Known Colab environment conflicts are recorded,
    # not silently treated as a clean environment.
    lines = [
        line.strip()
        for line in pip_result.stdout.splitlines()
        if line.strip()
    ]

    known_prefixes = (
        "ipython ",
        "google-colab ",
    )

    unexpected = [
        line for line in lines
        if not line.startswith(known_prefixes)
    ]

    if unexpected:
        raise RuntimeError(
            "Unexpected dependency conflicts require review:\n"
            + "\n".join(unexpected)
        )

    print(
        "Known Colab conflicts recorded. "
        "Continuing with import and unit tests."
    )

# ============================================================
# 6. FRESH PYTHON IMPORT TEST
# ============================================================

print("\n=== SCIENTIFIC IMPORT TEST ===")

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        """
import numpy
import pandas
import scipy
import sklearn
import joblib
import psutil
import google.auth
import googleapiclient.discovery

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)
print("IMPORT_TEST_PASS")
"""
    ],
    capture_output=True,
    text=True,
)

print(probe.stdout)
print(probe.stderr[-5000:])

if probe.returncode != 0:
    raise RuntimeError(
        "Scientific import test FAILED. "
        "Restart the runtime if packages were recently changed."
    )

print("Scientific imports: PASS")

# ============================================================
# 7. FIX WINDOWS ROOT PATH
# ============================================================

print("\n=== PORTABLE PATH CHECK ===")

training_file = (
    REPO / "huyen_code" / "ciciomt24_training.py"
)

if not training_file.is_file():
    raise FileNotFoundError(training_file)

source = training_file.read_text(encoding="utf-8")

old = 'ROOT = Path(r"D:\\COMPACT-MI-RFE")'
new = 'ROOT = Path(__file__).resolve().parents[1]'

if source.count(old) == 1:
    training_file.write_text(
        source.replace(old, new, 1),
        encoding="utf-8",
    )
    print("Windows ROOT path fixed.")

elif new in source:
    print("ROOT path already fixed.")

else:
    raise RuntimeError(
        "Unexpected ROOT definition. "
        "Inspect ciciomt24_training.py manually."
    )

original_dir = REPO / "CODE_ctu13"

if not original_dir.is_dir():
    raise FileNotFoundError(
        f"Missing CTU-13 scripts: {original_dir}"
    )

print("CTU-13 directory:", original_dir)

# ============================================================
# 8. REPOSITORY INVENTORY
# ============================================================

print("\n=== REPOSITORY INVENTORY ===")

plan = dependency_plan()

if plan["install_needed"]:
    raise RuntimeError(
        f"Scientific versions still differ: {plan}"
    )

inventory = repo_inventory(REPO)

if not inventory["pass"]:
    raise RuntimeError(
        f"Repository inventory failed: {inventory}"
    )

print("Repository inventory: PASS")

# ============================================================
# 9. UNIT TEST HELPER
# ============================================================

def run_tests(label, modules, cwd):
    print(f"\n=== {label} ===")

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "unittest",
            "-v",
            *modules,
        ],
        cwd=str(cwd),
        capture_output=True,
        text=True,
    )

    print(result.stdout[-4000:])
    print(result.stderr[-8000:])

    if result.returncode != 0:
        raise RuntimeError(
            f"{label} FAILED. Full training blocked."
        )

    print(f"{label}: PASS")

# ============================================================
# 10. SOURCE UNIT TESTS
# ============================================================

run_tests(
    "SOURCE UNIT TESTS",
    [
        "test_ciciomt24_training",
        "test_ciciomt24_split",
        "test_ciciomt24_sliding_window",
    ],
    REPO / "huyen_code",
)

# ============================================================
# 11. STAGE 6 ADAPTER TESTS
# ============================================================

run_tests(
    "STAGE 6 ADAPTER TESTS",
    [
        "colab.test_stage6_colab",
    ],
    REPO,
)

# ============================================================
# 12. SAVE UNIT TEST EVIDENCE
# ============================================================

def file_sha256(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()

UNIT_TEST_EVIDENCE = {
    "passed": True,
    "training_sha256": file_sha256(training_file),
}

print("\n====================================")
print("CELL 5: PASS")
print("SOURCE UNIT TESTS: PASS")
print("STAGE 6 ADAPTER TESTS: PASS")
print("GLOBAL PIP CHECK:", GLOBAL_PIP_CHECK_PASSED)
print(
    "Training SHA256:",
    UNIT_TEST_EVIDENCE["training_sha256"],
)
print("====================================")

print(
    "\nPython 3.13 uses a different NumPy profile "
    "from Stage 5. Bitwise parity is not claimed."
)


Actual Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Repository: /content/RFforNetwork

Selected scientific profile: {'numpy': '2.2.6', 'pandas': '2.3.3', 'scipy': '1.15.3', 'scikit-learn': '1.7.2', 'joblib': '1.5.3'}

=== SCIENTIFIC VERSION CHECK ===
numpy: 2.2.6 | required=2.2.6 | PASS
pandas: 2.3.3 | required=2.3.3 | PASS
scipy: 1.15.3 | required=1.15.3 | PASS
scikit-learn: 1.7.2 | required=1.7.2 | PASS
joblib: 1.5.3 | required=1.5.3 | PASS

=== GLOBAL DEPENDENCY AUDIT ===
ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement pandas==2.2.3, but you have pandas 2.3.3.


Known Colab conflicts recorded. Continuing with import and unit tests.

=== SCIENTIFIC IMPORT TEST ===
NumPy: 2.2.6
Pandas: 2.3.3
SciPy: 1.15.3
Scikit-learn: 1.7.2
Joblib: 1.5.3
IMPORT_TEST_PASS


Scientific imports: PASS

=== PORTABLE PATH CHECK ===
Windows ROOT path fixed.
CTU-13 directory: /content/RFforNetwork/CODE_ctu13

=== REPOSITORY INVENTORY ===
Repository inven

In [ ]:
# CELL 6 — Prefer the extracted WINDOWS_CPLUS folder; ZIP is optional
# Scientific validation runs in a fresh interpreter, not this Colab kernel.
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
import colab.dataset_validator as dataset_validator
from colab.drive_manager import discover_shared_dataset, download_archive
from colab.dataset_validator import (WINDOW_ROOT, inspect_archive, prepare_dataset,
    prepare_drive_folder_dataset, prepare_mounted_folder_dataset)

# Source manifests store Windows paths. This affects path lookup only, not data or statistics.
sw_file = REPO / 'huyen_code/ciciomt24_sliding_window.py'
sw_source = sw_file.read_text(encoding='utf-8')
old_stem = 'Path(segment.output_relative_path).stem'
if old_stem in sw_source:
    if sw_source.count(old_stem) != 2 or 'from pathlib import Path\n' not in sw_source:
        raise RuntimeError('Unexpected sliding-window source; no path patch applied')
    sw_source = sw_source.replace('from pathlib import Path\n',
                                  'from pathlib import Path, PureWindowsPath\n', 1)
    sw_file.write_text(sw_source.replace(old_stem,
                                         'PureWindowsPath(segment.output_relative_path).stem'),
                       encoding='utf-8')
    print('Portable Windows filename lookup: applied to Colab checkout')

validation_worker = r'''
import json, sys
from pathlib import Path
repo, windows = map(Path, sys.argv[1:3])
sys.path.insert(0, str(repo))
from colab.dataset_validator import validate_dataset
result = validate_dataset(windows, repo, all_headers=bool(int(sys.argv[3])))
print('__VALIDATED_JSON__' + json.dumps(result, default=str), flush=True)
'''

def validate_in_fresh_python(windows, repo, *, all_headers=True):
    process = subprocess.run(
        [sys.executable, '-u', '-c', validation_worker,
         str(repo), str(windows), str(int(all_headers))],
        cwd=str(repo), capture_output=True, text=True, check=False,
    )
    if process.returncode:
        print(process.stdout[-4000:])
        print(process.stderr[-8000:])
        raise RuntimeError('Isolated dataset validation failed; staging preserved')
    marker = '__VALIDATED_JSON__'
    lines = [line[len(marker):] for line in process.stdout.splitlines()
             if line.startswith(marker)]
    if len(lines) != 1:
        raise RuntimeError('Isolated validation returned no unique JSON result')
    return json.loads(lines[0])

# Dataset transfer stays unchanged; only its final validation is isolated.
dataset_validator.validate_dataset = validate_in_fresh_python

def recover_complete_mounted_stage(repo, source, output_root):
    stage = output_root.parent.with_name(output_root.parent.name + '.incomplete')
    if not stage.is_dir():
        return None
    plan_path = stage / 'transfer_plan.json'
    if not plan_path.is_file():
        raise FileNotFoundError(f'Unrecognized staged tree: {stage}')
    record = json.loads(plan_path.read_text(encoding='utf-8'))
    plan = {key: record[key] for key in ('source_kind', 'source_id', 'files')}
    digest = hashlib.sha256(json.dumps(plan, sort_keys=True,
                                       ensure_ascii=False).encode()).hexdigest()
    if digest != record.get('plan_sha256'):
        raise ValueError('Staged transfer plan hash mismatch')
    if plan['source_kind'] != 'mounted_folder' or plan['source_id'] != str(source.resolve()):
        raise ValueError('Staged source differs from this mounted folder')
    files = plan['files']
    if len(files) != 323 or sum(item['relative_path'].endswith('_windows.csv')
                                for item in files) != 322:
        raise ValueError('Staged plan does not contain 322 windows CSVs and summary')
    windows = stage / 'WINDOWS_CPLUS'
    for item in files:
        relative = Path(item['relative_path'])
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError(f'Unsafe staged path: {relative}')
        target = windows / relative
        partial = target.with_name(target.name + '.partial')
        if target.is_symlink() or partial.exists() or not target.is_file():
            print('Staged transfer incomplete; resume verified transfer:', target)
            return None
        if target.stat().st_size != item['size']:
            raise IOError(f'Staged size mismatch: {target}')
    if not (windows / 'window_run_summary.json').is_file():
        raise FileNotFoundError('Staged summary missing')
    if output_root.parent.exists():
        raise FileExistsError('Completed and staged dataset trees both exist')
    print('All 323 staged files found; validating in fresh Python without recopy')
    validated = validate_in_fresh_python(windows, repo)
    os.replace(stage, output_root.parent)
    return {'action': 'recovered_previously_verified_transfer',
            'transferred_files': 0, **validated,
            'dataset_path': str(output_root)}

DATASET_READY = False
if WINDOW_ROOT.exists():
    PREPARED = prepare_dataset(REPO, output_root=WINDOW_ROOT)
elif DRIVE_DATASET_PATH:
    source = Path(DRIVE_DATASET_PATH)
    if source.is_dir():
        windows_source = source/'WINDOWS_CPLUS' if (source/'WINDOWS_CPLUS').is_dir() else source
        PREPARED = recover_complete_mounted_stage(REPO, windows_source, WINDOW_ROOT)
        if PREPARED is None:
            PREPARED = prepare_mounted_folder_dataset(
                REPO, windows_source, output_root=WINDOW_ROOT)
    elif source.is_file():
        print('Optional ZIP inventory:',inspect_archive(source))
        PREPARED = prepare_dataset(REPO,archive=source,output_root=WINDOW_ROOT)
    else:
        raise FileNotFoundError(source)
else:
    found = discover_shared_dataset(SHARED_FOLDER_ID)  # user-authorized read-only API
    print('Shared entries:',found['children'])
    print('Audit folder present (not downloaded):',found['audit_folder_present'])
    if found['mode'] == 'folder':
        PREPARED = prepare_drive_folder_dataset(REPO,found['windows_folder']['id'],
                                                found['service'],output_root=WINDOW_ROOT)
    elif found['mode'] == 'zip_available':
        zips = found['zips']
        chosen = next((x for x in zips if x['id']==DRIVE_ARCHIVE_ID),None) if DRIVE_ARCHIVE_ID else (zips[0] if len(zips)==1 else None)
        if chosen is None: raise RuntimeError('Multiple ZIPs: specify DRIVE_ARCHIVE_ID; no folder found.')
        if (Path(chosen['name']).name != chosen['name'] or chr(92) in chosen['name']
                or chosen['name'] in ('.','..')): raise ValueError('Unsafe Drive ZIP filename')
        size = int(chosen.get('size') or 0)
        free = shutil.disk_usage('/content').free
        if size <= 0 or free < 2*size + 8*1024**3:
            raise OSError(f'ZIP disk admission FAIL: need {2*size+8*1024**3:,} bytes; have {free:,}')
        local_zip = Path('/content/datasets/.archive_downloads')/chosen['name']
        if local_zip.exists():
            if local_zip.stat().st_size != size: raise IOError('Existing ZIP differs; inspect manually.')
        else: download_archive(chosen['id'],local_zip)
        PREPARED = prepare_dataset(REPO,archive=local_zip,output_root=WINDOW_ROOT)
    else:
        raise FileNotFoundError('Shared folder has no WINDOWS_CPLUS folder or ZIP')
DATASET_READY = PREPARED['status'] == 'PASS_METADATA_SCHEMA'
print('Dataset path:',WINDOW_ROOT)
print('Configuration folders:',PREPARED['config_directories'],'CSV files:',PREPARED['csv_files'])
print('Manifest counts:',PREPARED['counts'],'Missing files:',PREPARED['missing_files'])
print('Schema validation:',PREPARED['status'],'DATASET_READY=',DATASET_READY)
print('Drive source was not moved or deleted; Test feature rows were not opened.')

[dataset 1/323] transfer FW_256_FS_16/test/Benign/sid001__sid001__Benign_test.pcap_Flow__test__FW256_FS16_windows.csv (2,126,314 bytes)
[dataset 2/323] transfer FW_256_FS_16/test/Benign/sid022__sid022__Benign_train.pcap_Flow__test__FW256_FS16_windows.csv (567,699 bytes)
[dataset 3/323] transfer FW_256_FS_16/test/MQTT-DDoS-Connect_Flood/sid002__sid002__MQTT-DDoS-Connect_Flood_test.pcap_Flow__test__FW256_FS16_windows.csv (238,501,481 bytes)
[dataset 4/323] transfer FW_256_FS_16/test/MQTT-DDoS-Publish_Flood/sid003__sid003__MQTT-DDoS-Publish_Flood_test.pcap_Flow__test__FW256_FS16_windows.csv (804,268 bytes)
[dataset 5/323] transfer FW_256_FS_16/test/MQTT-DDoS-Publish_Flood/sid024__sid024__MQTT-DDoS-Publish_Flood_train.pcap_Flow__test__FW256_FS16_windows.csv (4,648,015 bytes)
[dataset 6/323] transfer FW_256_FS_16/test/MQTT-DoS-Connect_Flood/sid004__sid004__MQTT-DoS-Connect_Flood_test.pcap_Flow__test__FW256_FS16_windows.csv (8,957,118 bytes)
[dataset 7/323] transfer FW_256_FS_16/test/MQTT-Do

AttributeError: 'numpy.ufunc' object has no attribute '__module__' and no __dict__ for setting new attributes

### Nếu Cell 6 lỗi sau khi chuyển 323/323 mục

Đừng chạy lại lượt chuyển hoặc xóa thư mục `.incomplete`. Chạy **RECOVER CELL 6** ngay bên dưới: cell này đếm 322 CSV đã chuyển, kiểm định trong tiến trình Python mới và chỉ promote thư mục staged khi PASS. Kernel Colab có thể bị lỗi import NumPy sau cài package dù tiến trình Python mới vẫn hoạt động. Nếu Cell 6 bản mới đã tự phục hồi thành công thì bỏ qua cell RECOVER.


In [ ]:

# RECOVER CELL 6 — Reuse all transferred CSVs
# No downloading, no copying, no training

from pathlib import Path
import subprocess
import sys

REPO = Path("/content/RFforNetwork")
FINAL = Path("/content/datasets/CICIoMT24/WINDOWS_CPLUS")
STAGED = Path("/content/datasets/CICIoMT24.incomplete/WINDOWS_CPLUS")

print("=== DATASET RECOVERY ===")
print("Final exists:", FINAL.is_dir())
print("Staged exists:", STAGED.is_dir())

if not FINAL.is_dir() and not STAGED.is_dir():
    raise FileNotFoundError(
        "Neither final nor staged dataset exists. "
        "Do not download again until paths are checked."
    )

# Fix Windows/Linux path interpretation if needed
sw_file = REPO / "huyen_code" / "ciciomt24_sliding_window.py"
source = sw_file.read_text(encoding="utf-8")

old = "Path(segment.output_relative_path).stem"
new = "PureWindowsPath(segment.output_relative_path).stem"

if old in source:
    if source.count(old) != 2:
        raise RuntimeError("Unexpected sliding-window source version")

    import_old = "from pathlib import Path\n"
    if import_old not in source:
        raise RuntimeError("Unexpected pathlib import")

    source = source.replace(
        import_old,
        "from pathlib import Path, PureWindowsPath\n",
        1
    )
    source = source.replace(old, new)
    sw_file.write_text(source, encoding="utf-8")
    print("Portable Windows filename fix: applied")
else:
    print("Portable Windows filename fix: already applied")

# Validate in a fresh Python process
recovery_script = r'''
from pathlib import Path
import os
import sys

REPO = Path("/content/RFforNetwork")
FINAL = Path("/content/datasets/CICIoMT24/WINDOWS_CPLUS")
STAGED = Path("/content/datasets/CICIoMT24.incomplete/WINDOWS_CPLUS")

sys.path.insert(0, str(REPO))

from colab.dataset_validator import validate_dataset

target = FINAL if FINAL.is_dir() else STAGED

csv_files = sum(
    1 for _ in target.rglob("*_windows.csv")
)

print("CSV files found:", csv_files, flush=True)

if csv_files != 322:
    raise RuntimeError(
        f"Expected 322 CSV files, found {csv_files}"
    )

if not (target / "window_run_summary.json").is_file():
    raise FileNotFoundError(
        target / "window_run_summary.json"
    )

print("Validating dataset...", flush=True)

result = validate_dataset(target, REPO)

assert result["status"] == "PASS_METADATA_SCHEMA"
assert result["csv_files"] == 322
assert result["features"] == 380
assert result["config_directories"] == 4

print("Validation: PASS", flush=True)

# Promote only after successful validation
if target == STAGED:
    stage_parent = STAGED.parent
    final_parent = FINAL.parent

    if final_parent.exists():
        raise FileExistsError(
            f"Final destination already exists: {final_parent}"
        )

    os.replace(stage_parent, final_parent)
    print("Staged dataset promoted successfully.")

assert FINAL.is_dir()

print("\n=== FINAL RESULT ===")
print("Status:", result["status"])
print("CSV files:", result["csv_files"])
print("Features:", result["features"])
print("Configurations:", result["config_directories"])
print("Final path:", FINAL)
print("Fingerprint:", result["dataset_fingerprint_sha256"])
print("DATASET_RECOVERY_PASS", flush=True)
'''

process = subprocess.run(
    [sys.executable, "-u", "-c", recovery_script],
    cwd=str(REPO),
    check=False,
)

if process.returncode != 0:
    raise RuntimeError(
        "Recovery validation failed. "
        "Transferred files were not deleted. "
        "Inspect the traceback before retrying."
    )

WINDOW_ROOT = FINAL
DATASET_READY = True

print("\nDATASET_READY =", DATASET_READY)
print("No dataset files downloaded or copied.")


=== DATASET RECOVERY ===
Final exists: False
Staged exists: True
Portable Windows filename fix: applied

DATASET_READY = True
No dataset files downloaded or copied.


In [ ]:

# CELL 7 — Validate C+ dataset in a clean Python process
# Avoid broken NumPy state in the current Colab kernel
# No download, no copy, no training

from pathlib import Path
import subprocess
import sys
import json

REPO = Path("/content/RFforNetwork")
WINDOW_ROOT = Path(
    "/content/datasets/CICIoMT24/WINDOWS_CPLUS"
)

if not WINDOW_ROOT.is_dir():
    raise FileNotFoundError(WINDOW_ROOT)

print("=== STAGE 6 DATASET AUDIT ===")
print("Dataset:", WINDOW_ROOT)
print("Running validation in isolated Python process...")

worker = r"""
import sys
import json
from pathlib import Path

repo = Path(sys.argv[1])
window_root = Path(sys.argv[2])

sys.path.insert(0, str(repo))

from colab.dataset_validator import validate_dataset

result = validate_dataset(window_root, repo)

print(
    "__AUDIT_JSON__" +
    json.dumps(result, default=str, ensure_ascii=False)
)
"""

process = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        worker,
        str(REPO),
        str(WINDOW_ROOT),
    ],
    cwd=str(REPO),
    capture_output=True,
    text=True,
)

if process.returncode != 0:
    print(process.stdout[-4000:])
    print(process.stderr[-8000:])
    raise RuntimeError("Dataset audit failed")

marker = "__AUDIT_JSON__"

payload = next(
    (
        line[len(marker):]
        for line in process.stdout.splitlines()
        if line.startswith(marker)
    ),
    None,
)

if payload is None:
    raise RuntimeError("Audit JSON not returned")

DATA_AUDIT = json.loads(payload)

DATASET_READY = (
    DATA_AUDIT.get("status") == "PASS_METADATA_SCHEMA"
    and DATA_AUDIT.get("csv_files") == 322
    and DATA_AUDIT.get("features") == 380
    and DATA_AUDIT.get("config_directories") == 4
)

print("\n=== VALIDATION RESULTS ===")
print("Status:", DATA_AUDIT.get("status"))
print("CSV files:", DATA_AUDIT.get("csv_files"))
print("Features:", DATA_AUDIT.get("features"))
print("Configurations:", DATA_AUDIT.get("config_directories"))
print("Counts:", DATA_AUDIT.get("counts"))
print("Label origin:", DATA_AUDIT.get("label_origin"))
print(
    "Test feature rows read:",
    DATA_AUDIT.get("test_feature_rows_read")
)
print("DATASET_READY:", DATASET_READY)

if not DATASET_READY:
    raise RuntimeError(
        "Dataset validation did not pass"
    )

print("\n================================")
print("CELL 7: PASS")
print("DATASET READY: TRUE")
print("No dataset files downloaded or copied")
print("================================")


=== STAGE 6 DATASET AUDIT ===
Dataset: /content/datasets/CICIoMT24/WINDOWS_CPLUS
Running validation in isolated Python process...

=== VALIDATION RESULTS ===
Status: PASS_METADATA_SCHEMA
CSV files: 322
Features: 380
Configurations: 4
Counts: {'FW_256_FS_8': {'train': 1348317, 'val': 449343, 'test': 449251}, 'FW_256_FS_16': {'train': 674169, 'val': 224677, 'test': 224631}, 'FW_512_FS_8': {'train': 1347101, 'val': 448735, 'test': 448563}, 'FW_512_FS_16': {'train': 673559, 'val': 224373, 'test': 224287}}
Label origin: SCENARIO_PROXY
Test feature rows read: 0
DATASET_READY: True

CELL 7: PASS
DATASET READY: TRUE
No dataset files downloaded or copied


## Stage 7.1 — SVM Baseline / MI-RFE-SVM technical probe

Output RF hiển thị ở các cell cũ là **lịch sử** và được giữ nguyên. Cell 8 đã được đặt lại `dry_run` để tránh chạy lại thí nghiệm khi mở notebook. Muốn đo SVM: bảo đảm Cell 4A báo nguồn Train-only sẵn sàng, tại Cell 8 chọn `baseline_svm` hoặc `mi_rfe_svm`, đặt `RUN_NAME` mới, giữ `MODE=dry_run`, bật `RUN_TECHNICAL_SVM_PROBE`; sau đó chạy Cell 9 và 10. Cell 8 tự chạy unit tests Stage 7.1 trong tiến trình riêng. Cell 9 báo full-SVM preflight `FAIL` là **đúng thiết kế**; Cell 10 vẫn được phép chạy probe giới hạn trên Train. Không dùng Validation/Test để benchmark, không xuất Accuracy chính thức và không bật full SVM trước khi có quyết định Stage 7.2.


In [ ]:
# CELL 8 — Select exactly one experiment/config; default is DRY RUN
EXPERIMENT = "mi_rfe_rf"  #@param ["mi_rfe_rf", "baseline_svm", "mi_rfe_svm"]
FW = "256"  #@param [256, 512]
FS = 8  #@param [8, 16]
RUN_NAME = "mi_rfe_rf_FW256_FS8_run02"  #@param {type:"string"}
MODE = "dry_run"  #@param ["dry_run", "full_train"]
CONFIRMATION = ""  #@param {type:"string"}
RUN_TECHNICAL_SVM_PROBE = False  #@param {type:"boolean"}

import os
import subprocess
import sys

from colab.experiment_launcher import RunSpec
from colab.drive_manager import DRIVE_ROOT

SPEC = RunSpec(EXPERIMENT, f'FW_{FW}_FS_{FS}', RUN_NAME)
SPEC.validate()
LOCAL_RUNS = Path('/content/stage6_runs')

if RUN_TECHNICAL_SVM_PROBE and SPEC.experiment == 'mi_rfe_rf':
    raise ValueError('Technical SVM probe cannot run for RF')

SVM_STAGE71_TESTS_PASS = False
if SPEC.experiment in ('baseline_svm', 'mi_rfe_svm'):
    if MODE != 'dry_run':
        raise RuntimeError('Full SVM is blocked at Stage 7.1; select dry_run')
    probe_source = REPO / 'huyen_code/ciciomt24_svm_probe.py'
    launcher_source = REPO / 'colab/experiment_launcher.py'
    test_source = REPO / 'huyen_code/test_ciciomt24_svm_probe.py'
    if not (probe_source.is_file() and launcher_source.is_file() and test_source.is_file()):
        raise FileNotFoundError('Run Cell 4A to sync all three Stage 7.1 files')
    if ('stage7.1.train_only.svm_probe.v1' not in probe_source.read_text(encoding='utf-8')
            or 'from ciciomt24_svm_probe import run_probe' not in launcher_source.read_text(encoding='utf-8')):
        raise RuntimeError('Old SVM code detected; sync current files in Cell 4A')
    test_env = os.environ.copy()
    test_env['PYTHONPATH'] = f"{REPO}:{REPO / 'huyen_code'}"
    test_run = subprocess.run(
        [sys.executable, '-m', 'unittest', 'discover',
         '-s', str(REPO / 'huyen_code'),
         '-p', 'test_ciciomt24_svm_probe.py', '-v'],
        cwd=str(REPO), env=test_env, capture_output=True, text=True,
    )
    print(test_run.stdout[-2000:])
    print(test_run.stderr[-6000:])
    if test_run.returncode:
        raise RuntimeError('Stage 7.1 Colab unit tests failed; SVM probe blocked')
    SVM_STAGE71_TESTS_PASS = True
    print('Stage 7.1 SVM unit tests: PASS; full SVM still blocked')

print({
    'experiment': SPEC.experiment, 'config': SPEC.config,
    'run': SPEC.run_name, 'mode': MODE,
    'technical_svm_probe': RUN_TECHNICAL_SVM_PROBE,
    'local_output': str(LOCAL_RUNS), 'drive_output': str(DRIVE_ROOT),
})
print('Full training requires Cell 10 and the exact confirmation phrase.')

{'experiment': 'mi_rfe_rf', 'config': 'FW_256_FS_8', 'run': 'mi_rfe_rf_FW256_FS8_run02', 'mode': 'full_train', 'local_output': '/content/stage6_runs', 'drive_output': '/content/drive/MyDrive/COMPACT-MI-RFE'}
Full train requires CELL 10 and the exact confirmation phrase.


In [ ]:

# CELL 9 — SAFE PREFLIGHT IN FRESH PYTHON PROCESS
# Avoid NumPy import errors in the Colab kernel
# No dataset copy, no download, no model.fit

from pathlib import Path
import subprocess
import sys
import json

print("=== STAGE 6 PREFLIGHT ===")

# 1. Verify prerequisites
if not DATASET_READY:
    raise RuntimeError(
        "Dataset validation has not passed."
    )

if not UNIT_TEST_EVIDENCE.get("passed", False):
    raise RuntimeError(
        "Unit tests have not passed."
    )

REPO = Path(REPO)
WINDOW_ROOT = Path(WINDOW_ROOT)
LOCAL_RUNS = Path(LOCAL_RUNS)
DRIVE_ROOT = Path(DRIVE_ROOT)

if not WINDOW_ROOT.is_dir():
    raise FileNotFoundError(WINDOW_ROOT)

# 2. Prepare experiment configuration
PROBE_PATH = LOCAL_RUNS / (
    SPEC.run_name + "_svm_technical_probe.json"
)

payload = {
    "repo": str(REPO),
    "windows": str(WINDOW_ROOT),
    "local_runs": str(LOCAL_RUNS),
    "drive_root": str(DRIVE_ROOT),
    "spec": {
        "experiment": SPEC.experiment,
        "config": SPEC.config,
        "run_name": SPEC.run_name,
    },
    "unit_test_evidence": UNIT_TEST_EVIDENCE,
    "svm_probe_path": (
        str(PROBE_PATH)
        if PROBE_PATH.exists()
        else None
    ),
}

# 3. Execute original preflight in fresh process
worker = r"""
import sys
import json
from pathlib import Path

data = json.load(sys.stdin)

repo = Path(data["repo"])
sys.path.insert(0, str(repo))

from colab.experiment_launcher import (
    RunSpec,
    preflight,
)

spec = RunSpec(**data["spec"])

probe = (
    Path(data["svm_probe_path"])
    if data["svm_probe_path"]
    else None
)

result = preflight(
    repo,
    Path(data["windows"]),
    Path(data["local_runs"]),
    Path(data["drive_root"]),
    spec,
    unit_test_evidence=data["unit_test_evidence"],
    svm_probe_path=probe,
)

print(
    "__PREFLIGHT_JSON__"
    + json.dumps(result, default=str)
)
"""

print("Python:", sys.version.split()[0])
print("Dataset:", WINDOW_ROOT)
print("Experiment:", SPEC.experiment)
print("Config:", SPEC.config)
print("Run:", SPEC.run_name)
print("\nRunning isolated preflight...")

process = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        worker,
    ],
    input=json.dumps(payload),
    cwd=str(REPO),
    capture_output=True,
    text=True,
    check=False,
)

# 4. Check worker execution
if process.returncode != 0:
    print(process.stdout[-4000:])
    print(process.stderr[-8000:])
    raise RuntimeError(
        "Preflight worker failed. "
        "No training was started."
    )

# 5. Recover preflight result
marker = "__PREFLIGHT_JSON__"

result_line = next(
    (
        line[len(marker):]
        for line in process.stdout.splitlines()
        if line.startswith(marker)
    ),
    None,
)

if result_line is None:
    raise RuntimeError(
        "Preflight worker did not return JSON."
    )

PREFLIGHT = json.loads(result_line)

# 6. Display results
print("\n=== PREFLIGHT RESULTS ===")

print("PREFLIGHT:", PREFLIGHT["status"])

print("\nCHECKS:")
for name, passed in PREFLIGHT["checks"].items():
    print(
        f"  {name}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nRAM GUARD:")
print(PREFLIGHT.get("ram_guard"))

print("\nLOCAL DISK REQUIRED BYTES:")
print(PREFLIGHT.get("local_disk_required_bytes"))

print("\nSAFETY:")
print(
    "Test feature rows read:",
    PREFLIGHT.get("test_feature_rows_read")
)
print(
    "model.fit calls:",
    PREFLIGHT.get("model_fit_calls")
)

# 7. Enforce preflight safety
if PREFLIGHT["status"] == "PASS":
    print("\nPREFLIGHT PASS")
else:
    failed = [
        key
        for key, value in PREFLIGHT["checks"].items()
        if not value
    ]
    print("\nPREFLIGHT FAIL")
    print("Failed checks:", failed)
    print("Full training is BLOCKED.")

print("\nCELL 9 COMPLETED")


=== STAGE 6 PREFLIGHT ===
Python: 3.13.15
Dataset: /content/datasets/CICIoMT24/WINDOWS_CPLUS
Experiment: mi_rfe_rf
Config: FW_256_FS_8
Run: mi_rfe_rf_FW256_FS8_run02

Running isolated preflight...

=== PREFLIGHT RESULTS ===
PREFLIGHT: PASS

CHECKS:
  repository_files: PASS
  python_supported: PASS
  selected_scientific_profile: PASS
  unit_tests_current_code: PASS
  approved_original_scripts: PASS
  dataset_manifest_schema: PASS
  no_existing_local_run: PASS
  no_existing_drive_run: PASS
  drive_mounted: PASS
  prior_fs16_not_rerun: PASS
  ram_admission: PASS
  local_disk_admission: PASS

RAM GUARD:
{'policy': 'stage54_empirical_scaled_FS16_peak_plus_max_2GiB_or_20pct', 'required_available_bytes': 25792855276, 'available_bytes': 52465545216, 'guard_is_not_proven_peak_bound': True}

LOCAL DISK REQUIRED BYTES:
8589934592

SAFETY:
Test feature rows read: 0
model.fit calls: 0

PREFLIGHT PASS

CELL 9 COMPLETED


In [ ]:

# CELL 10 — Isolated RF training or bounded Stage 7.1 SVM probe
# Fix NumPy kernel import error
# Keep all safety checks, checkpoints and Drive outputs
# No dataset download or copy

from pathlib import Path
import subprocess
import tempfile
import json
import sys

RUN_RESULT = None

REPO = Path(REPO).resolve()
WINDOW_ROOT = Path(WINDOW_ROOT).resolve()
LOCAL_RUNS = Path(LOCAL_RUNS)
DRIVE_ROOT = Path(DRIVE_ROOT)

PROBE_PATH = LOCAL_RUNS / (
    SPEC.run_name + "_svm_technical_probe.json"
)

print('=== STAGE 6 RF / STAGE 7.1 SVM ===')
print("Python:", sys.version.split()[0])
print("Experiment:", SPEC.experiment)
print("Config:", SPEC.config)
print("Run:", SPEC.run_name)
print("Mode:", MODE)
print("Dataset:", WINDOW_ROOT)

# 1. Safety checks
if not DATASET_READY:
    raise RuntimeError("Dataset validation has not passed")

if not UNIT_TEST_EVIDENCE.get("passed"):
    raise RuntimeError("Unit tests have not passed")

if not WINDOW_ROOT.is_dir():
    raise FileNotFoundError(WINDOW_ROOT)

SPEC.validate()

if MODE not in ("dry_run", "full_train"):
    raise ValueError("Unsupported training mode")

if SPEC.experiment in ('baseline_svm', 'mi_rfe_svm'):
    if MODE != 'dry_run':
        raise RuntimeError('Stage 7.1 blocks full SVM training')
    if not globals().get('SVM_STAGE71_TESTS_PASS', False):
        raise RuntimeError('Run Cell 8 and pass Stage 7.1 tests first')

if RUN_TECHNICAL_SVM_PROBE and SPEC.experiment == "mi_rfe_rf":
    raise ValueError("SVM probe cannot be used for RF")

if MODE == "full_train":
    if PREFLIGHT["status"] != "PASS":
        raise RuntimeError("Preflight FAIL: training blocked")

    required = (
        f"RUN {SPEC.experiment} "
        f"{SPEC.config} {SPEC.run_name}"
    )

    if CONFIRMATION != required:
        raise PermissionError(
            f"Incorrect confirmation. Expected: {required}"
        )

    # Never overwrite earlier runs or checkpoints
    occupied = [
        LOCAL_RUNS / SPEC.run_name,
        LOCAL_RUNS / (SPEC.run_name + ".incomplete"),
        DRIVE_ROOT / "experiment_results" / SPEC.run_name,
        DRIVE_ROOT / "experiment_results" /
            (SPEC.run_name + ".incomplete"),
        DRIVE_ROOT / "checkpoints" /
            (SPEC.run_name + ".incomplete"),
        DRIVE_ROOT / "environment" /
            (SPEC.run_name + ".json"),
    ]

    existing = [str(p) for p in occupied if p.exists()]

    if existing:
        raise FileExistsError(
            "Run artifacts already exist; inspect them "
            "before choosing a new run name:\n"
            + "\n".join(existing)
        )

# 2. Pass configuration to the clean process
payload = {
    "repo": str(REPO),
    "windows": str(WINDOW_ROOT),
    "local_runs": str(LOCAL_RUNS),
    "drive_root": str(DRIVE_ROOT),
    "spec": {
        "experiment": SPEC.experiment,
        "config": SPEC.config,
        "run_name": SPEC.run_name,
    },
    "mode": MODE,
    "confirmation": CONFIRMATION,
    "unit_test_evidence": UNIT_TEST_EVIDENCE,
    "svm_probe": bool(RUN_TECHNICAL_SVM_PROBE),
    "probe_path": str(PROBE_PATH),
}

# 3. Fresh Python worker
worker = r"""
import json
import sys
from pathlib import Path

data = json.loads(
    Path(sys.argv[1]).read_text(encoding="utf-8")
)

repo = Path(data["repo"])
sys.path.insert(0, str(repo))

# Import scientific packages in fresh process
import numpy
import pandas
import sklearn
import scipy
import joblib

print("Scientific imports: PASS", flush=True)
print("NumPy:", numpy.__version__, flush=True)

from colab.experiment_launcher import (
    RunSpec,
    preflight,
    run_one,
    run_technical_svm_probe,
)

spec = RunSpec(**data["spec"])
spec.validate()

windows = Path(data["windows"])
local_runs = Path(data["local_runs"])
drive_root = Path(data["drive_root"])
probe_path = Path(data["probe_path"])

if data["svm_probe"]:
    local_runs.mkdir(parents=True, exist_ok=True)

    probe_result = run_technical_svm_probe(
        repo, windows, spec, probe_path
    )

    print(
        "__PROBE_JSON__" +
        json.dumps(probe_result, default=str),
        flush=True
    )

    audit = preflight(
        repo, windows, local_runs, drive_root, spec,
        unit_test_evidence=data["unit_test_evidence"],
        svm_probe_path=probe_path
    )

    print(
        "Updated preflight:",
        audit["status"],
        audit["checks"],
        flush=True
    )

    if data["mode"] == "full_train":
        raise RuntimeError(
            "Stage 7.1 does not authorize full SVM. "
            "Review probe resources and request Stage 7.2 approval."
        )

elif data["mode"] == "full_train":
    print("\nStarting official training...", flush=True)

    result = run_one(
        repo,
        windows,
        local_runs,
        drive_root,
        spec,
        unit_test_evidence=data["unit_test_evidence"],
        confirmation=data["confirmation"],
        svm_probe_path=(
            probe_path if probe_path.exists() else None
        ),
    )

    print(
        "__RUN_RESULT_JSON__" +
        json.dumps(result, default=str),
        flush=True
    )

else:
    print("Dry run only. No model.fit called.", flush=True)
"""

# 4. Launch worker and stream logs in real time
if MODE == "dry_run" and not RUN_TECHNICAL_SVM_PROBE:
    print("\nDRY RUN ONLY — No training started.")

else:
    print("\nLaunching isolated Python process...")
    print('Bounded SVM probe only.' if RUN_TECHNICAL_SVM_PROBE
          else 'Do not interrupt during full RF training.')

    with tempfile.TemporaryDirectory(
        prefix="stage6_launch_"
    ) as temp_dir:

        config_file = Path(temp_dir) / "launch.json"

        config_file.write_text(
            json.dumps(payload),
            encoding="utf-8"
        )

        process = subprocess.Popen(
            [
                sys.executable,
                "-u",
                "-c",
                worker,
                str(config_file),
            ],
            cwd=str(REPO),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            errors="replace",
        )

        result_json = None
        probe_json = None

        for line in process.stdout:
            clean = line.strip()

            if clean.startswith("__RUN_RESULT_JSON__"):
                result_json = clean.split(
                    "__RUN_RESULT_JSON__", 1
                )[1]
            elif clean.startswith("__PROBE_JSON__"):
                probe_json = clean.split(
                    "__PROBE_JSON__", 1
                )[1]
            else:
                print(line, end="", flush=True)

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Probe/training worker failed (exit {return_code}). "
            "Inspect local and Drive checkpoints before "
            "retrying. No automatic restart is performed."
        )

    if result_json is not None:
        RUN_RESULT = json.loads(result_json)

        print("\n=== OFFICIAL RUN RESULT ===")
        print(json.dumps(
            RUN_RESULT,
            indent=2,
            ensure_ascii=False
        ))

        print("\nCELL 10: TRAINING COMPLETED")

    elif probe_json is not None:
        print("\nSVM TECHNICAL PROBE RESULT:")
        print(json.loads(probe_json))

    elif MODE == "full_train":
        raise RuntimeError(
            "Worker exited without an official result."
        )


=== STAGE 6 — OFFICIAL RUN ===
Python: 3.13.15
Experiment: mi_rfe_rf
Config: FW_256_FS_8
Run: mi_rfe_rf_FW256_FS8_run02
Mode: full_train
Dataset: /content/datasets/CICIoMT24/WINDOWS_CPLUS

Launching isolated Python process...
Do not interrupt during full training.

Scientific imports: PASS
NumPy: 2.2.6

Starting official training...
[0.0s] run_started stage=run_started rss=0.180GiB available=48.890GiB {"config": "FW_256_FS_8", "experiment": "mi_rfe_rf", "resume_supported": false}
[0.0s] split_load_started stage=split_load_started rss=0.180GiB available=48.890GiB {"split": "train", "expected_rows": 1348317, "files": 45}
[0.1s] source_file_loaded stage=source_file_loaded rss=0.185GiB available=48.888GiB {"split": "train", "source_id": 6, "source_rows": 82, "loaded_rows": 82, "expected_rows": 1348317}
[0.1s] source_file_loaded stage=source_file_loaded rss=0.188GiB available=48.885GiB {"split": "train", "source_id": 11, "source_rows": 44, "loaded_rows": 126, "expected_rows": 1348317}
[0.2s

In [ ]:
# CELL 10A — Read SVM technical JSON only; never open window/Test CSVs
import json

if SPEC.experiment in ('baseline_svm', 'mi_rfe_svm'):
    if not PROBE_PATH.is_file():
        print('No SVM probe JSON yet:', PROBE_PATH)
    else:
        technical = json.loads(PROBE_PATH.read_text(encoding='utf-8'))
        print('Probe:', PROBE_PATH)
        print('Status:', technical.get('status'))
        print('Validation/Test CSV rows read:',
              technical.get('validation_csv_rows_read'),
              technical.get('test_csv_rows_read'))
        print('Full training authorized:', technical.get('full_training_authorized'))
        for run in technical.get('runs', []):
            print('Sample:', run.get('sample_rows'),
                  'worker:', run.get('status'),
                  'peak tree RSS bytes:', run.get('peak_process_tree_rss_bytes_sampled'),
                  'stop:', run.get('supervisor_stop_reason'))
            for fit in run.get('fits', []):
                measurement = fit.get('fit', {})
                print('  k/features:', fit.get('features'),
                      'fit seconds:', measurement.get('seconds'),
                      'support vectors:', measurement.get('support_vectors'),
                      'converged:', measurement.get('converged'))
        print('Technical timing only — not Accuracy, not a full-SVM gate.')
else:
    print('Cell 10A is only for SVM technical probes.')

In [ ]:
# CELL 11 — Verify publication; optionally promote an ALREADY COMPLETED checkpoint
import json
import hashlib
from pathlib import Path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

import json
RECOVER_COMPLETED_CHECKPOINT = False  #@param {type:"boolean"}
if RECOVER_COMPLETED_CHECKPOINT:
    from colab.checkpoint_manager import publish_completed_checkpoint
    from colab.dataset_validator import EXPECTED
    recovered_path = publish_completed_checkpoint(DRIVE_ROOT,SPEC.run_name,EXPECTED[SPEC.config])
    print('Promoted existing completed checkpoint without fit or Test evaluation:',recovered_path)
if RUN_RESULT is None:
    print("No new official result to save; no Drive result folder created by this cell.")
else:
    result_dir = Path(RUN_RESULT['drive_result'])
    manifest = json.loads((result_dir/'artifact_manifest.json').read_text())
    for name, expected in manifest['files'].items():
        path = result_dir/name
        if path.stat().st_size != expected['bytes'] or sha256(path) != expected['sha256']:
            raise IOError(f"Drive result artifact failed verification: {name}")
    print("Drive artifacts verified:",result_dir)
    print("Environment:",DRIVE_ROOT/'environment'/(SPEC.run_name+'.json'))
    print("Checkpoints:",DRIVE_ROOT/'checkpoints'/(SPEC.run_name+'.incomplete'))
    print("No Test prediction/evaluation was repeated by this cell.")

Drive artifacts verified: /content/drive/MyDrive/COMPACT-MI-RFE/experiment_results/mi_rfe_rf_FW256_FS8_run02
Environment: /content/drive/MyDrive/COMPACT-MI-RFE/environment/mi_rfe_rf_FW256_FS8_run02.json
Checkpoints: /content/drive/MyDrive/COMPACT-MI-RFE/checkpoints/mi_rfe_rf_FW256_FS8_run02.incomplete
No Test prediction/evaluation was repeated by this cell.


In [ ]:
# CELL 12 — Read result.json only; never load Test CSV/model for summary
from colab.results_summary import completed_results, previous_artifacts_status
rows = completed_results(DRIVE_ROOT)
print("Previous FS16 artifacts:",previous_artifacts_status(DRIVE_ROOT))
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for row in rows: print(row)
print("SCENARIO_PROXY; Benign is rare; capture/session independence remains unproven.")

Previous FS16 artifacts: {'FW_512_FS_16': {'status': 'PREVIOUS_ARTIFACTS_MISSING', 'result_json': []}, 'FW_256_FS_16': {'status': 'PREVIOUS_ARTIFACTS_MISSING', 'result_json': []}}


,run,experiment,config,status,accuracy,macro_f1,balanced_accuracy,benign_precision,benign_recall,benign_f1,attack_precision,attack_recall,attack_f1,selected_features,elapsed_seconds,peak_rss_bytes_sampled,confusion_matrix_0_benign_1_attack,label_origin,result_json
0,mi_rfe_rf_FW256_FS8_run02,mi_rfe_rf,FW_256_FS_8,PASS,0.999604,0.951214,0.911266,1.0,0.822532,0.902626,0.999603,1.0,0.999801,20,7389.470855,21382422528,"[[825, 178], [0, 448248]]",SCENARIO_PROXY,/content/drive/MyDrive/COMPACT-MI-RFE/experime...
1,mi_rfe_rf_FW512_FS8_run01,mi_rfe_rf,FW_512_FS_8,PASS,0.999842,0.980315,0.962194,1.0,0.924388,0.960708,0.999841,1.0,0.999921,17,7871.308281,21467930624,"[[868, 71], [0, 447624]]",SCENARIO_PROXY,/content/drive/MyDrive/COMPACT-MI-RFE/experime...


SCENARIO_PROXY; Benign is rare; capture/session independence remains unproven.


## Stage 7.2 — bounded Train-only SVM scaling probe

Optional technical probe only. First sync the updated repository manually, keep Stage 6 `MODE="dry_run"`, confirm the C+ dataset and run the tests cell. Then enable exactly one experiment at a time. The probe samples only Train at 16k/32k/64k, never reads Validation/Test CSV rows, and never authorizes full SVM. Do not use its internal holdout as a scientific result.

In [ ]:
# STAGE 7.2 TESTS — opt-in, no training
STAGE72_RUN_TESTS = False
STAGE72_TESTS_PASS = False
if STAGE72_RUN_TESTS:
    import subprocess, sys
    from pathlib import Path
    stage72_repo = Path('/content/RFforNetwork').resolve()
    for relative, pattern in [('huyen_code', 'test_ciciomt24_svm_scaling_probe.py'), ('huyen_code', 'test_ciciomt24_svm_probe.py'), ('colab', 'test_stage6_colab.py')]:
        completed = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(stage72_repo / relative), '-p', pattern, '-v'], cwd=str(stage72_repo))
        if completed.returncode != 0:
            raise RuntimeError(f'Stage 7.2 test failure: {pattern}; probe blocked')
    STAGE72_TESTS_PASS = True
    print('Stage 7.2 tests PASS; technical probe remains opt-in.')
else:
    print('Stage 7.2 tests disabled; probe remains blocked.')

In [ ]:
# STAGE 7.2 CONFIG — one experiment/config, disabled by default
from pathlib import Path
STAGE72_ENABLE = False
STAGE72_EXPERIMENT = 'baseline_svm'  # baseline_svm, then mi_rfe_svm in a separate run
STAGE72_CONFIG = 'FW_256_FS_8'
STAGE72_RUN_NAME = 'baseline_svm_FW256_FS8_scaling01'
STAGE72_SIZES = (16000, 32000, 64000)
STAGE72_TIMEOUT_PER_SIZE = 1200
STAGE72_MAX_RSS_GIB = 12
STAGE72_LOCAL_RUNS = Path('/content/stage72_runs')
STAGE72_REPO = Path('/content/RFforNetwork').resolve()
STAGE72_WINDOWS = Path('/content/datasets/CICIoMT24/WINDOWS_CPLUS')
STAGE72_MANIFEST = STAGE72_REPO / 'configs' / 'cplus_manifest'
print({'enabled': STAGE72_ENABLE, 'experiment': STAGE72_EXPERIMENT, 'config': STAGE72_CONFIG, 'sizes': STAGE72_SIZES, 'run': STAGE72_RUN_NAME})

In [ ]:
# STAGE 7.2 RUN — opt-in; starts one bounded supervisor, not full training
if not STAGE72_ENABLE:
    print('Stage 7.2 probe disabled; no worker started.')
else:
    import re, subprocess, sys
    if globals().get('MODE') != 'dry_run' or not globals().get('DATASET_READY', False):
        raise RuntimeError('Require Stage 6 MODE=dry_run and DATASET_READY=True')
    if not globals().get('STAGE72_TESTS_PASS', False):
        raise RuntimeError('Run Stage 7.2 tests successfully before probe')
    if STAGE72_EXPERIMENT not in ('baseline_svm', 'mi_rfe_svm') or STAGE72_CONFIG not in ('FW_256_FS_8', 'FW_256_FS_16', 'FW_512_FS_8', 'FW_512_FS_16'):
        raise ValueError('Unknown Stage 7.2 experiment/config')
    if not re.fullmatch(r'[A-Za-z0-9_-]{1,80}', STAGE72_RUN_NAME):
        raise ValueError('Unsafe Stage 7.2 run name')
    if tuple(STAGE72_SIZES) != (16000, 32000, 64000):
        raise ValueError('This notebook uses the approved 16k/32k/64k sequence')
    script = STAGE72_REPO / 'huyen_code' / 'ciciomt24_svm_scaling_probe.py'
    if not script.is_file() or not (STAGE72_MANIFEST / 'split_manifest.csv').is_file():
        raise FileNotFoundError('Sync the Stage 7.2 code and approved manifest before running')
    if not STAGE72_WINDOWS.is_dir():
        raise FileNotFoundError('C+ windows missing')
    STAGE72_LOCAL_RUNS.mkdir(parents=True, exist_ok=True)
    output = STAGE72_LOCAL_RUNS / (STAGE72_RUN_NAME + '.json')
    if output.exists() or list(STAGE72_LOCAL_RUNS.glob(STAGE72_RUN_NAME + '.n*.json')):
        raise FileExistsError('Stage 7.2 output exists; choose a fresh run name')
    command = [sys.executable, str(script), 'probe', '--enable-scaling-probe', '--repo', str(STAGE72_REPO), '--windows', str(STAGE72_WINDOWS), '--manifest-root', str(STAGE72_MANIFEST), '--config', STAGE72_CONFIG, '--experiment', STAGE72_EXPERIMENT, '--output', str(output), '--sizes', *map(str, STAGE72_SIZES), '--seed', '42', '--timeout-per-size', str(STAGE72_TIMEOUT_PER_SIZE), '--max-rss-gib', str(STAGE72_MAX_RSS_GIB)]
    completed = subprocess.run(command, cwd=str(STAGE72_REPO))
    if completed.returncode != 0:
        raise RuntimeError(f'Stage 7.2 technical probe stopped (exit {completed.returncode}); preserve incomplete artifacts')
    print('Technical-only JSON:', output, '; full SVM still BLOCKED')

In [ ]:
# STAGE 7.2 REVIEW — JSON only; no CSV/model reads or fit
STAGE72_RUN_REVIEW = False
if STAGE72_RUN_REVIEW:
    import sys
    sys.path.insert(0, str(STAGE72_REPO / 'huyen_code'))
    from ciciomt24_svm_scaling_probe import render_resource_review
    baseline = STAGE72_LOCAL_RUNS / 'baseline_svm_FW256_FS8_scaling01.json'
    compact = STAGE72_LOCAL_RUNS / 'mi_rfe_svm_FW256_FS8_scaling01.json'
    for artifact in (baseline, compact):
        if not artifact.is_file():
            raise FileNotFoundError(artifact)
    review = render_resource_review(baseline, compact)
    print(review)
else:
    print('Stage 7.2 JSON-only review disabled.')

In [ ]:
# STAGE 7.2 BACKUP — small JSON/report only, opt-in, no overwrite
STAGE72_BACKUP_TO_DRIVE = False
if STAGE72_BACKUP_TO_DRIVE:
    import shutil
    from pathlib import Path
    target = Path('/content/drive/MyDrive/COMPACT-MI-RFE/technical_probes/stage72')
    if not Path('/content/drive/MyDrive').is_dir():
        raise FileNotFoundError('Google Drive is not mounted')
    artifacts = sorted(STAGE72_LOCAL_RUNS.glob('*.json'))
    if not artifacts:
        raise FileNotFoundError('No Stage 7.2 JSON artifacts')
    if target.exists() and any((target / item.name).exists() for item in artifacts):
        raise FileExistsError('Drive artifact already exists; no overwrite')
    target.mkdir(parents=True, exist_ok=True)
    for item in artifacts:
        with (target / item.name).open('xb') as stream, item.open('rb') as source:
            shutil.copyfileobj(source, stream)
    print('Copied JSON artifacts to:', target)
else:
    print('Stage 7.2 Drive backup disabled.')